# Day 2 — PySpark: String Column Functions
### Target: Data Engineer Interviews

> **Roadmap Day:** 2 · **Date:** Tuesday, June 16, 2026  
> **Note:** Same problems as SQL Day 2 — same data, same business logic, PySpark DataFrame API.

Run each cell in order.

---
## Setup — Environment & SparkSession

In [1]:
import os, sys

os.environ['JAVA_HOME']             = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

print("Environment set. Run next cell to start Spark.")

Environment set. Run next cell to start Spark.


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit,
    upper, lower, initcap,
    trim, ltrim, rtrim,
    regexp_replace, split, substring,
    length, instr,
    concat, concat_ws,
    when, coalesce
)
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

spark = SparkSession.builder \
    .appName("Day02_StringFunctions") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} ready.")

Spark 3.5.6 ready.


---
## 1. The customers DataFrame — Same Messy Data as SQL Day 2

In [3]:
customers_data = [
    (1,  '  Alice Johnson  ', 'ALICE@Gmail.COM   ', '  +91-98765-43210 '),
    (2,  'bob smith',          'bob@yahoo.com',      '9876100002'),
    (3,  'CAROL WHITE',        ' carol@hotmail.com', '(098) 761-00003'),
    (4,  'David Brown',        'david@gmail.com',    '987.610.0004'),
    (5,  'eve nair  ',         'EVE@GMAIL.COM',      '+919876100005'),
    (6,  'Frank Singh',        'frank@outlook.com',  '98-7610-0006'),
    (7,  'GRACE PATEL',        'GRACE@yahoo.com ',   '9876100007'),
    (8,  '  henry das',        'henry@gmail.com',    '(987) 610-0008'),
    (9,  'Irene Verma',        'irene@gmail.COM',    '9876100009  '),
    (10, 'jack KUMAR',         'JACK@hotmail.com',   '987-610-0010'),
]

customers_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("full_name",   StringType(),  True),
    StructField("email",       StringType(),  True),
    StructField("phone",       StringType(),  True),
])

df = spark.createDataFrame(customers_data, schema=customers_schema)
print("customers created — 10 rows. Preview:")
df.show(3, truncate=False)

customers created — 10 rows. Preview:
+-----------+-----------------+------------------+------------------+
|customer_id|full_name        |email             |phone             |
+-----------+-----------------+------------------+------------------+
|1          |  Alice Johnson  |ALICE@Gmail.COM   |  +91-98765-43210 |
|2          |bob smith        |bob@yahoo.com     |9876100002        |
|3          |CAROL WHITE      | carol@hotmail.com|(098) 761-00003   |
+-----------+-----------------+------------------+------------------+
only showing top 3 rows



---
## 2. upper() / lower() / initcap() — Case Standardisation

Same as SQL `UPPER()`, `LOWER()`, `INITCAP()`.  
**Import from `pyspark.sql.functions`** — these are column functions, not Python built-ins.

In [4]:
# lower + trim — standard email normalisation
df.select(
    col("customer_id"),
    col("email"),
    lower(trim(col("email"))).alias("email_clean")
).show(5, truncate=False)

+-----------+------------------+-----------------+
|customer_id|email             |email_clean      |
+-----------+------------------+-----------------+
|1          |ALICE@Gmail.COM   |alice@gmail.com  |
|2          |bob@yahoo.com     |bob@yahoo.com    |
|3          | carol@hotmail.com|carol@hotmail.com|
|4          |david@gmail.com   |david@gmail.com  |
|5          |EVE@GMAIL.COM     |eve@gmail.com    |
+-----------+------------------+-----------------+
only showing top 5 rows



In [6]:
# initcap + trim — title-case the full_name
df.select(
    col("customer_id"),
    col("full_name"),
    initcap(
        trim(
            col("full_name")
        )
    ).alias("name_clean")
).show(5, truncate=False)

+-----------+-----------------+-------------+
|customer_id|full_name        |name_clean   |
+-----------+-----------------+-------------+
|1          |  Alice Johnson  |Alice Johnson|
|2          |bob smith        |Bob Smith    |
|3          |CAROL WHITE      |Carol White  |
|4          |David Brown      |David Brown  |
|5          |eve nair         |Eve Nair     |
+-----------+-----------------+-------------+
only showing top 5 rows



In [7]:
# Case-insensitive filter — same pattern as SQL WHERE TRIM(LOWER(email)) = ...
df.filter(lower(trim(col("email"))) == "alice@gmail.com") \
  .select("customer_id") \
  .show(truncate=False)

+-----------+
|customer_id|
+-----------+
|1          |
+-----------+



---
## 3. trim() / ltrim() / rtrim() — Whitespace Removal

Equivalent to SQL `TRIM`, `LTRIM`, `RTRIM`.  
Spark's `trim()` only strips spaces — use `regexp_replace` to strip other characters.

In [8]:
# Show rows that have leading/trailing whitespace in full_name
df.filter(col("full_name") != trim(col("full_name"))) \
  .select(
      col("customer_id"),
      col("full_name"),
      trim(col("full_name")).alias("name_trimmed")
  ).show(truncate=False)

+-----------+-----------------+-------------+
|customer_id|full_name        |name_trimmed |
+-----------+-----------------+-------------+
|1          |  Alice Johnson  |Alice Johnson|
|5          |eve nair         |eve nair     |
|8          |  henry das      |henry das    |
+-----------+-----------------+-------------+



In [9]:
# ltrim / rtrim demonstration
demo = spark.createDataFrame([("  hello world  ",)], ["s"])
demo.select(
    col("s").alias("original"),
    trim(col("s")).alias("trim_both"),
    ltrim(col("s")).alias("ltrim_only"),
    rtrim(col("s")).alias("rtrim_only")
).show(truncate=False)

+---------------+-----------+-------------+-------------+
|original       |trim_both  |ltrim_only   |rtrim_only   |
+---------------+-----------+-------------+-------------+
|  hello world  |hello world|hello world  |  hello world|
+---------------+-----------+-------------+-------------+



---
## 4. regexp_replace() — Pattern-Based Replacement

`regexp_replace(col, pattern, replacement)` — replaces ALL matches by default (no 'g' flag needed).  
Most powerful string cleaning tool in PySpark.

In [10]:
# Remove all non-digit characters from phone
df.select(
    col("customer_id"),
    col("phone"),
    regexp_replace(trim(col("phone")), r"[^0-9]", "").alias("phone_clean")
).show(truncate=False)

+-----------+------------------+------------+
|customer_id|phone             |phone_clean |
+-----------+------------------+------------+
|1          |  +91-98765-43210 |919876543210|
|2          |9876100002        |9876100002  |
|3          |(098) 761-00003   |09876100003 |
|4          |987.610.0004      |9876100004  |
|5          |+919876100005     |919876100005|
|6          |98-7610-0006      |9876100006  |
|7          |9876100007        |9876100007  |
|8          |(987) 610-0008    |9876100008  |
|9          |9876100009        |9876100009  |
|10         |987-610-0010      |9876100010  |
+-----------+------------------+------------+



In [11]:
# Collapse multiple spaces to one, then trim
demo = spark.createDataFrame([("  alice   johnson  ",)], ["s"])
demo.select(
    col("s").alias("raw"),
    trim(regexp_replace(col("s"), r"\s+", " ")).alias("normalised")
).show(truncate=False)

+-------------------+-------------+
|raw                |normalised   |
+-------------------+-------------+
|  alice   johnson  |alice johnson|
+-------------------+-------------+



---
## 5. split() — Split String Into Array

`split(col, pattern)` returns an **ArrayType** column.  
Access elements with `[index]` — **0-indexed** (unlike SQL's SPLIT_PART which is 1-indexed).

In [12]:
# Split full_name into first and last name
# Note: split()[0] = first element (0-indexed, unlike SQL SPLIT_PART's 1-indexed)
df.withColumn("name_clean", initcap(trim(col("full_name")))) \
  .select(
      col("customer_id"),
      initcap(trim(col("full_name"))).alias("full_name"),
      split(initcap(trim(col("full_name"))), r"\s+")[0].alias("first_name"),
      split(initcap(trim(col("full_name"))), r"\s+")[1].alias("last_name")
  ).show(5, truncate=False)

+-----------+-------------+----------+---------+
|customer_id|full_name    |first_name|last_name|
+-----------+-------------+----------+---------+
|1          |Alice Johnson|Alice     |Johnson  |
|2          |Bob Smith    |Bob       |Smith    |
|3          |Carol White  |Carol     |White    |
|4          |David Brown  |David     |Brown    |
|5          |Eve Nair     |Eve       |Nair     |
+-----------+-------------+----------+---------+
only showing top 5 rows



In [13]:
# Gotcha: split is 0-indexed — contrast with SQL SPLIT_PART (1-indexed)
demo = spark.createDataFrame([("a|b|c|d",)], ["log_line"])
demo.select(
    col("log_line"),
    split(col("log_line"), r"\|")[0].alias("field1"),   # 'a'
    split(col("log_line"), r"\|")[1].alias("field2"),   # 'b'
    split(col("log_line"), r"\|")[2].alias("field3")    # 'c'
).show(truncate=False)

+--------+------+------+------+
|log_line|field1|field2|field3|
+--------+------+------+------+
|a|b|c|d |a     |b     |c     |
+--------+------+------+------+



---
## 6. substring() — Extract Part of a String

`substring(col, pos, len)` — **1-based** position, same as SQL.  
Returns empty string if pos is beyond the string length.

In [14]:
# Extract date parts from a date-string column
demo = spark.createDataFrame([("2024-01-15",)], ["full_date"])
demo.select(
    col("full_date"),
    substring(col("full_date"), 1, 4).alias("year"),   # '2024'
    substring(col("full_date"), 6, 2).alias("month"),  # '01'
    substring(col("full_date"), 9, 2).alias("day"),    # '15'
    substring(col("full_date"), 1, 7).alias("ym"),     # '2024-01'
).show(truncate=False)

+----------+----+-----+---+-------+
|full_date |year|month|day|ym     |
+----------+----+-----+---+-------+
|2024-01-15|2024|01   |15 |2024-01|
+----------+----+-----+---+-------+



In [16]:
# Extract domain from email using instr + substring
df_clean = df.withColumn("email", lower(trim(col("email"))))

df_clean.select(
    col("customer_id"),
    col("email"),
    substring(col("email"), instr(col("email"), "@") + 1, 100).alias("domain")
).show(2, truncate=False)

PySparkTypeError: [NOT_ITERABLE] Column is not iterable.

---
## 7. concat() / concat_ws() — Combine Strings

| Function | NULL behaviour |
|----------|----------------|
| `concat(a, b)` | If ANY argument is NULL → result is NULL |
| `concat_ws(sep, a, b)` | Skips NULLs, inserts separator between non-NULLs |

In [ ]:
# Build display label: "Alice Johnson <alice@gmail.com>"
df.withColumn(
    "display_label",
    concat(
        initcap(trim(col("full_name"))),
        lit(" <"),
        lower(trim(col("email"))),
        lit(">")
    )
).select("customer_id", "display_label").show(3, truncate=False)

+-----------+---------------------------+
|customer_id|display_label              |
+-----------+---------------------------+
|1          |Alice Johnson <alice@gmail.com>|
|2          |Bob Smith <bob@yahoo.com>  |
|3          |Carol White <carol@hotmail.com>|
+-----------+---------------------------+
only showing top 3 rows


In [ ]:
# NULL difference: concat vs concat_ws
demo = spark.createDataFrame([(None, "world", "foo")], ["a", "b", "c"])
demo.select(
    concat(col("a"), lit(", "), col("b")).alias("concat_result"),    # NULL — a is null
    concat_ws(", ", col("a"), col("b"), col("c")).alias("concat_ws_result")  # 'world, foo'
).show(truncate=False)

+-------------------+-------------------+
|concat_result      |concat_ws_result   |
+-------------------+-------------------+
|null               |world, foo         |
+-------------------+-------------------+


---
## 8. length() / instr() — Length and Position

`length(col)` — number of characters.  
`instr(col, substr)` — 1-based position, **0** if not found (same as SQL `POSITION`).

In [ ]:
# Validate phone length after cleaning
df.withColumn(
    "phone_clean",
    regexp_replace(trim(col("phone")), r"[^0-9]", "")
).withColumn(
    "digit_len",
    length(regexp_replace(trim(col("phone")), r"[^0-9]", ""))
).select("customer_id", "phone", "phone_clean", "digit_len") \
 .show(3, truncate=False)

+-----------+--------------------+------------+----------+
|customer_id|phone               |phone_clean |digit_len |
+-----------+--------------------+------------+----------+
|1          |  +91-98765-43210   |9198765432  |10        |
|3          |(098) 761-00003     |0987610000  |10        |
|5          |+919876100005       |91987610000 |11        |
+-----------+--------------------+------------+----------+
only showing top 3 rows


In [ ]:
# instr — find position of '@' in email
df.withColumn("email", lower(trim(col("email")))) \
  .withColumn("at_pos", instr(col("email"), "@")) \
  .withColumn("has_at", instr(col("email"), "@") > 0) \
  .select("customer_id", "email", "at_pos", "has_at") \
  .show(2, truncate=False)

+-----------+--------------------+------+--------+
|customer_id|email               |at_pos|has_at  |
+-----------+--------------------+------+--------+
|1          |alice@gmail.com     |6     |true    |
|2          |bob@yahoo.com       |4     |true    |
+-----------+--------------------+------+--------+
only showing top 2 rows


---
## 9. String Predicates — contains, startswith, endswith, rlike

These are **Column methods**, not functions — call them on the column object directly.

In [ ]:
df_lower = df.withColumn("email", lower(trim(col("email"))))

# contains — email has '@gmail'
print("gmail customers:")
df_lower.filter(col("email").contains("@gmail")) \
        .select("customer_id", "email") \
        .show(truncate=False)

gmail customers:
+-----------+--------------------+
|customer_id|email               |
+-----------+--------------------+
|1          |alice@gmail.com     |
|5          |eve@gmail.com       |
|8          |henry@gmail.com     |
|9          |irene@gmail.com     |
+-----------+--------------------+


In [ ]:
# NOT contains — find emails missing '@'
print("No '@' in email:")
df_lower.filter(~col("email").contains("@")) \
        .select("customer_id", "email") \
        .show(truncate=False)

# endswith — find .com emails
print(".com emails:")
df_lower.filter(col("email").endswith(".com")) \
        .select("customer_id", "email") \
        .show(3, truncate=False)

No '@' in email:
+-----------+------+
|customer_id|email |
+-----------+------+
+-----------+------+

.com emails:
+-----------+--------------------+
|customer_id|email               |
+-----------+--------------------+
|1          |alice@gmail.com     |
|2          |bob@yahoo.com       |
|3          |carol@hotmail.com   |
+-----------+--------------------+
only showing top 3 rows


---
## 10. Day 2 Problems — Official Solutions

Same 3 problems as SQL Day 2 — implement in PySpark DataFrame API.

In [ ]:
# Problem 1 (Easy): Standardise emails to lowercase + trim
print("Problem 1: Standardise emails")
df.select(
    col("customer_id"),
    col("email"),
    lower(trim(col("email"))).alias("email_clean")
).orderBy("customer_id").show(truncate=False)

Problem 1: Standardise emails
+-----------+--------------------+--------------------+
|customer_id|email               |email_clean         |
+-----------+--------------------+--------------------+
|1          |ALICE@Gmail.COM     |alice@gmail.com     |
|2          |bob@yahoo.com       |bob@yahoo.com       |
|3          | carol@hotmail.com  |carol@hotmail.com   |
|4          |david@gmail.com     |david@gmail.com     |
|5          |EVE@GMAIL.COM       |eve@gmail.com       |
|6          |frank@outlook.com   |frank@outlook.com   |
|7          |GRACE@yahoo.com     |grace@yahoo.com     |
|8          |henry@gmail.com     |henry@gmail.com     |
|9          |irene@gmail.COM     |irene@gmail.com     |
|10         |JACK@hotmail.com    |jack@hotmail.com    |
+-----------+--------------------+--------------------+


In [ ]:
# Problem 2 (Easy): Split full_name into first and last name
print("Problem 2: Split full_name into first and last name")
df.select(
    col("customer_id"),
    initcap(trim(col("full_name"))).alias("full_name"),
    split(initcap(trim(col("full_name"))), r"\s+")[0].alias("first_name"),
    split(initcap(trim(col("full_name"))), r"\s+")[1].alias("last_name")
).orderBy("customer_id").show(truncate=False)

Problem 2: Split full_name into first and last name
+-----------+--------------+----------+---------+
|customer_id|full_name     |first_name|last_name|
+-----------+--------------+----------+---------+
|1          |Alice Johnson |Alice     |Johnson  |
|2          |Bob Smith     |Bob       |Smith    |
|3          |Carol White   |Carol     |White    |
|4          |David Brown   |David     |Brown    |
|5          |Eve Nair      |Eve       |Nair     |
|6          |Frank Singh   |Frank     |Singh    |
|7          |Grace Patel   |Grace     |Patel    |
|8          |Henry Das     |Henry     |Das      |
|9          |Irene Verma   |Irene     |Verma    |
|10         |Jack Kumar    |Jack      |Kumar    |
+-----------+--------------+----------+---------+


In [ ]:
# Problem 3 (Medium): Find customers with non-standard phones + show cleaned version
print("Problem 3: Find customers with non-10-digit phones")
phone_clean = regexp_replace(trim(col("phone")), r"[^0-9]", "")

df.withColumn("phone_clean", phone_clean) \
  .filter(length(col("phone_clean")) != 10) \
  .select(
      col("customer_id"),
      initcap(trim(col("full_name"))).alias("full_name"),
      col("phone"),
      col("phone_clean")
  ).orderBy("customer_id").show(truncate=False)

Problem 3: Find customers with non-10-digit phones
+-----------+----------+--------------------+------------+
|customer_id|full_name |phone               |phone_clean |
+-----------+----------+--------------------+------------+
|1          |Alice Johnson|  +91-98765-43210 |9198765432  |
|5          |Eve Nair  |+919876100005       |91987610000 |
+-----------+----------+--------------------+------------+


---
## 11. SparkSQL Alternative — Same 3 Problems via spark.sql()

Register the DataFrame as a temp view and solve with SQL strings.

In [ ]:
df.createOrReplaceTempView("customers")
print("Temp view registered.")

Temp view registered.


In [ ]:
# Problem 1 via SparkSQL
spark.sql("""
    SELECT
        customer_id,
        email,
        LOWER(TRIM(email)) AS email_clean
    FROM customers
    ORDER BY customer_id
""").show(truncate=False)

In [ ]:
# Problem 2 via SparkSQL
spark.sql("""
    SELECT
        customer_id,
        INITCAP(TRIM(full_name))                         AS full_name,
        SPLIT(INITCAP(TRIM(full_name)), ' ')[0]          AS first_name,
        SPLIT(INITCAP(TRIM(full_name)), ' ')[1]          AS last_name
    FROM customers
    ORDER BY customer_id
""").show(truncate=False)

In [ ]:
# Problem 3 via SparkSQL
spark.sql("""
    SELECT
        customer_id,
        INITCAP(TRIM(full_name))                         AS full_name,
        phone,
        REGEXP_REPLACE(TRIM(phone), '[^0-9]', '')        AS phone_clean
    FROM customers
    WHERE LENGTH(REGEXP_REPLACE(TRIM(phone), '[^0-9]', '')) != 10
    ORDER BY customer_id
""").show(truncate=False)

---
## 12. PySpark vs SQL Mapping — String Functions

| SQL | PySpark DataFrame API | SparkSQL |
|-----|----------------------|----------|
| `UPPER(col)` | `upper(col("x"))` | `UPPER(x)` |
| `LOWER(col)` | `lower(col("x"))` | `LOWER(x)` |
| `INITCAP(col)` | `initcap(col("x"))` | `INITCAP(x)` |
| `TRIM(col)` | `trim(col("x"))` | `TRIM(x)` |
| `REGEXP_REPLACE(col, p, r, 'g')` | `regexp_replace(col("x"), p, r)` | `REGEXP_REPLACE(x, p, r)` |
| `SPLIT_PART(col, sep, 1)` | `split(col("x"), sep)[0]` | `SPLIT(x, sep)[0]` |
| `SUBSTRING(col FROM 1 FOR 4)` | `substring(col("x"), 1, 4)` | `SUBSTRING(x, 1, 4)` |
| `CONCAT(a, b)` | `concat(col("a"), col("b"))` | `CONCAT(a, b)` |
| `CONCAT_WS(',', a, b)` | `concat_ws(",", col("a"), col("b"))` | `CONCAT_WS(',', a, b)` |
| `LENGTH(col)` | `length(col("x"))` | `LENGTH(x)` |
| `POSITION('@' IN col)` | `instr(col("x"), "@")` | `INSTR(x, '@')` |
| `col LIKE '%@%'` | `col("x").contains("@")` | `x LIKE '%@%'` |
| `col LIKE 'A%'` | `col("x").startswith("A")` | `x LIKE 'A%'` |
| `col ~ '^[0-9]+'` | `col("x").rlike("^[0-9]+")` | `x RLIKE '^[0-9]+'` |

**Key difference:** `SPLIT_PART` is 1-indexed (SQL), `split()[0]` is 0-indexed (PySpark).

In [ ]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.
